In [30]:
import pandas as pd
from gensim.models import Word2Vec
import jieba

# 读取你的 train.csv，用正确的列名 comment
df = pd.read_csv('train.csv')

# 对 comment 列分词（你的评论在这一列，绝对正确）
sentences = []
for text in df['comment']:
    words = jieba.lcut(str(text).strip())
    sentences.append(words)

print(f"✅ 数据加载完成！共 {len(sentences)} 条评论")

✅ 数据加载完成！共 10000 条评论


In [31]:
# 训练 Skip-Gram 模型（sg=1，符合任务要求）
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,  # 保留所有词，避免KeyError
    sg=1,
    workers=4
)

print("✅ Skip-Gram 模型训练完成！词表总词数：", len(model.wv.index_to_key))

✅ Skip-Gram 模型训练完成！词表总词数： 11780


In [32]:
# 你的评论里有“环境”这个词，直接用
vec_env = model.wv['环境']

print("===== 任务2 结果 =====")
print("“环境”的词向量：")
print(vec_env)
print("向量形状：", vec_env.shape)

===== 任务2 结果 =====
“环境”的词向量：
[-0.02372511 -0.02980005 -0.22334154  0.05689607 -0.09926636 -0.7729802
 -0.0597663   1.0557975  -1.0425678  -0.3975554  -0.24725142 -0.45277876
  0.12154839  0.6684082  -0.06199295 -0.1750057  -0.06101251  0.08561064
 -0.20524076 -1.5176886   0.21254508  0.06640926  0.11599806 -0.4083238
 -0.17779735  0.34421754 -0.5536664   0.18552123 -0.32802758 -0.22563927
  0.39326957  0.17027578  0.19459185 -0.8715619  -0.10283847  0.3714129
  0.48571938 -0.30452952 -0.03942234 -0.45614514  0.07794473 -0.26014
 -0.46807578 -0.27893883  0.15256774 -0.2945408  -0.11364955  0.271722
  0.5301974   0.05864393  0.08083473 -0.22780459  0.0657241   0.28781122
  0.23921289  0.13960876 -0.25276104 -0.07266632 -0.24112426  0.13570018
  0.09584058 -0.35221565  0.01164094  0.20598316 -0.12196395  0.44876963
  0.39929414 -0.01728345 -0.55958444  0.48378837 -0.1845199  -0.13669023
 -0.04379177 -0.3522213   0.22967854  0.1539335   0.16965204  0.26880792
 -0.00343411 -0.3375277  -0.08

In [33]:
# 你的评论里“好吃”是高频词，结果准确
top3 = model.wv.most_similar('好吃', topn=3)

print("===== 任务3 结果 =====")
for word, score in top3:
    print(f"{word}: {score:.4f}")

===== 任务3 结果 =====
入味: 0.8736
好看: 0.8710
美味: 0.8679


In [34]:
# 计算任务要求的两组相似度
sim1 = model.wv.similarity('好吃', '美味')
sim2 = model.wv.similarity('好吃', '蟑螂')

print("===== 任务4 结果 =====")
print(f"好吃 ↔ 美味：{sim1:.4f}")
print(f"好吃 ↔ 蟑螂：{sim2:.4f}")

===== 任务4 结果 =====
好吃 ↔ 美味：0.8679
好吃 ↔ 蟑螂：0.4541


In [35]:
# 向量运算，完全符合任务要求
result = model.wv.most_similar(
    positive=['餐厅', '聚会'],
    negative=['安静'],
    topn=1
)

print("===== 任务5 结果 =====")
print(f"餐厅 + 聚会 - 安静 = {result[0][0]}")

===== 任务5 结果 =====
餐厅 + 聚会 - 安静 = 门口
